# Force and Motion


In [1]:
import sympy as sp
from sympy.physics.mechanics import *
import sympy.physics.vector as spv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
from sympy.physics.vector.printing import init_vprinting
init_vprinting(use_latex='mathjax')

In [2]:
def reference_frame(
    frame: str,
    x=r'\imath', y=r'\jmath', z=r'k'
) -> ReferenceFrame:
    return ReferenceFrame(
        frame, latexs=(
            fr'\; {{}}^\mathcal {frame} \hat {x}',
            fr'\;{{}}^\mathcal {frame} \hat {y}',
            fr'\: {{}}^\mathcal {frame} \hat {z}'
        )
    )


def reference_frame_circular(name: str, angle=r"theta"):
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")


In [3]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_circular("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)

r = sp.Symbol("r", positive=True)
vec_r = r * P.x
vec_theta = P.y
assert vec_r.dot(vec_theta) == 0

PN_values = {P.x: P.x.express(N), P.y: P.y.express(N)}
display(
    Math(
        r"\boxed{"
        + r" \text{Polar to Cartesian: } \\[5pt] \\"
        + r"\begin{aligned}"
        + sp.latex(P.x)
        + " &= "
        + sp.latex(PN_values[P.x])
        + r" \\ "
        + sp.latex(P.y)
        + " &= "
        + sp.latex(PN_values[P.y])
        + r"\end{aligned}"
        + r"}"
    )
)

# Velocity
vec_v = vec_r.dt(N)
vec_a = vec_v.dt(N)
display(
    Math(
        r" \text{Velocity and Acceleration in Polar Coordinates: } \\[5pt] \\"
        + r"\boxed{"
        + r"\begin{aligned}"
        + r"\vec{v}"
        + " &= "
        + spv.vlatex(vec_v)
        + r" \\ "
        + r"\vec{a}"
        + " &= "
        + spv.vlatex(vec_a)
        + r"\end{aligned}"
        + r"}"
    )
)

# Constaint angular velocity omega, angular acceleration 0
omega = sp.Symbol("omega", positive=True)

values = {theta.diff(t): omega, theta.diff(t, 2): 0}
display(
    Math(
        r"\text{Uniform Circular Motion:} \\[5pt] \\"
        + r"\boxed{"
            + r"\begin{aligned}"
                + r"\vec{v} &= "
                + spv.vlatex(vec_v.subs(values))
                + r" && \text{where } \omega = "+ spv.vlatex(theta.diff(t)) + r" \text{ is constant} \\ "
                + r"\vec{a} &= "
                + spv.vlatex(vec_a.subs(values))
                + r" && \text{where } " + spv.vlatex(theta.diff(t, 2)) + r" = 0" + r" \\ "
            + r"\end{aligned}"
        + r"}"
    ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Show that for Uniform Circular Motion

$$\boxed{\frac{d \hat{\theta}}{dt} = -\frac{d \theta}{dt}\hat{r}}$$

In [4]:
t = dynamicsymbols._t
N = reference_frame("N")
P = reference_frame_circular("P")
theta = dynamicsymbols("theta")
P.orient_axis(N, theta, N.z)


display(Math(r"\frac{d\hat{\theta}}{dt} =" + spv.vlatex(P.y.diff(t, N).express(P).simplify())))

<IPython.core.display.Math object>

## 11.2 WE - Car on a Banked Turn

A car of mass $m$ is turning on a banked curve of angle $\phi$ with respect to the horizontal. The curve is icy and friction between the tires and the surface is negligible. The curve has a radius $r$. What is the speed $v$ at which the car can turn safely? Express your answer in terms of $m$, $\phi$, and $r$.

In [5]:
m, phi, r, g = sp.symbols("m phi r g", positive=True)
m, phi, r, g

t = dynamicsymbols._t
theta = dynamicsymbols("theta")

N = reference_frame("N")

P = reference_frame_circular("P")
P.orient_axis(N, theta, N.z)

NmB = sp.Symbol("N_mB", positive=True)

vec_FmE = m * g * (-P.y)
vec_NmB = NmB * (sp.cos(sp.pi / 2 + phi) * P.x + sp.cos(phi) * P.y)
vec_F_total = vec_FmE + vec_NmB

vec_r = r * P.x
vec_v = vec_r.diff(t, N)
vec_a = vec_v.diff(t, N).subs(theta.diff(t, 2), 0)

display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"\vec{r} &= "
        + spv.vlatex(vec_r)
        + r"\\"
        + r"\vec{v} &= "
        + spv.vlatex(vec_v)
        + r"\\"
        + r"\vec{a} &= "
        + spv.vlatex(vec_a) + r"\;,&& \text{where } \ddot{\theta} = 0"
        + r"\end{aligned}}"
    )
)

N2Leq = sp.Eq(vec_F_total.to_matrix(N), m * vec_a.to_matrix(N))
result = sp.solve(N2Leq, [NmB, theta.diff(t)], dict=True)

result_vec_v = [
    vec_v.subs(_).subs(theta.diff(t, 2), 0).magnitude().simplify() for _ in result
]
display(
    Math(
        r"\boxed{\begin{aligned}"
        + r"|\vec{v}| &= "
        + spv.vlatex(result_vec_v[0].simplify())
        + r"\end{aligned}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## PS.3.1 WE - Orbital Circular Motion

A person on a spherical asteroid of mass $m_1$ and radius $R$, sees a small satellite of mass $m_2$ orbiting the asteroid in a circular orbit of period $T$.

Express your answers in terms of some or all of the following: $m_1$, $m_2$, $\pi$, $T$, and the universal gravitational constant $G$.

(Part a) What is the radius $r_{sat}$ of the satellite’s orbit?

(Part b) What is the magnitude of the velocity of the satellite?

Part (c) If the asteroid rotates about its axis with a period $T_{a}$, at what radius must the satellite orbit the asteroid so that the satellite appears stationary to the person on the asteroid?

Express your answer in terms of some or all of the following: $m_1$, $m_2$, $\pi$, $R$, $T$, and the universal gravitational constant $G$.

Express your answer in terms of some or all of the following: $m_1$, $m_2$, $R$, $T$, and the universal gravitational constant $G$.



In [6]:
# (Part a) What is the radius $r_{sat}$ of the satellite’s orbit?

m1, m2, R, T, G, r = sp.symbols("m1 m2 R T G r", real=True, positive=True)
m1, m2, R, T, G, r

t = dynamicsymbols._t
theta = dynamicsymbols("theta")
theta_d = dynamicsymbols("theta", 1)
theta_dd = dynamicsymbols("theta", 2)
display(Math(r"\frac{d\theta}{dt} = " + spv.vlatex(theta_d)))
display(Math(r"\frac{d^2\theta}{dt^2} = " + spv.vlatex(theta_dd)))

N = reference_frame("N")

P = reference_frame_circular("P")
P.orient_axis(N, theta, N.z)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
vec_F21 = G * m1 * m2 / r**sp.Integer(2)*(-P.x)
display(Math(r"\vec{F}_{21} = " + spv.vlatex(vec_F21)))
vec_F21_N = vec_F21.express(N)

vec_r = r * P.x
display(Math(r"\vec{r} = " + spv.vlatex(vec_r)))
vec_v = msubs(vec_r.dt(N), {theta_d: 2*sp.pi/T})
vec_a = msubs(vec_v.dt(N), {theta_d: 2*sp.pi/T}, {theta_dd: 0})

display(Math(r"\vec{v} = " + spv.vlatex(vec_v)))
display(Math(r"\vec{a} = " + spv.vlatex(vec_a)))

N2Leq = sp.Eq((m2 * vec_a.to_matrix(N)), vec_F21_N.to_matrix(N))
display(Math(r"\text{Newton's 2nd Law: } " + spv.vlatex(N2Leq)))


result = sp.solve(N2Leq, [r], dict=True)[0][r]

display(Math(r"\text{Solution for } r_{sat}: " + spv.vlatex(result)))
display(
    Math(r"\boxed{\text{Compact form: } r_{sat} = " + r"\sqrt[3]{\frac{G m_1 T^2}{4 \pi^2}}}")
)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
# (Part b) What is the magnitude of the velocity of the satellite?

display(Math(f'v = ' + spv.vlatex(vec_v.magnitude().subs(r, result).simplify())))
display(Math(r'\text{Compact form: v = }' + r'\sqrt[3]{\frac{2 \pi m_1 G}{T}}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem Set 3

[Problem Set 3](./Problem%20Set%203.pdf)

## Problem 3.1

![Problem 3.1](../figures/PS0301-RotatingHoop.jpg)

In [9]:
display(Markdown("## Part (a)"))

# Define the symbols for mass, radius, and angular velocity
m, R, Omega = sp.symbols("m R Omega", real=True, positive=True)

# Define the dynamicsymbol for time
t = dynamicsymbols._t

# Define the dynamicsymbol for angular velocity, its first and second derivatives
omega = dynamicsymbols("omega")
omega_d = dynamicsymbols("omega", 1)
omega_dd = dynamicsymbols("omega", 2)

# Define the dynamicsymbol for angular position, its first and second derivatives
theta = dynamicsymbols("theta")
theta_d = dynamicsymbols("theta", 1)
theta_dd = dynamicsymbols("theta", 2)

# Create reference frames

# Reference frame 'N' with no transformations
N = reference_frame("N")

# Reference frame 'P' with rotational motion about the y-axis
P = reference_frame(
    "P",
    x=r"\omega",  # Angular velocity
    y=r"e_y",  # Unit vector in the y-direction
    z=r"r",  # Position vector
)

# Orient the frame with respect to the axis of rotation
P.orient_axis(N, omega, N.y)  

# Reference frame 'Q' with rotational motion about the z-axis
Q = reference_frame(
    "Q",
    x=r"r",  # Position vector
    y=r"\theta",  # Angular position
    z=r"e_z",  # Unit vector in the z-direction
)

# Orient the frame with respect to the axis of rotation
Q.orient_axis(P, 3 * sp.pi / 2 + theta, P.z)  

# Define the values of angular velocity and its derivatives
values_omega = {omega_d: Omega, omega_dd: 0}
values_theta = {theta_d: 0, theta_dd: 0}



## Part (a)

In [10]:

display(Markdown("#### Position, Velocity, and Acceleration"))

vec_r = R * Q.x  # Vector from the rotation axis to the particle
vec_v = vec_r.dt(N).subs(values_omega).subs(values_theta)  # Velocity in the rotating frame
vec_a = vec_v.dt(N).subs(values_omega).subs(values_theta)  # Acceleration in the rotating frame
display(Math(r"{{}}^\mathcal {Q}\vec{r} = " + spv.vlatex(vec_r)))
display(Math(r"{{}}^\mathcal {N}\vec{v} = " + spv.vlatex(vec_v)))
display(Math(r"{{}}^\mathcal {N}\vec{a} = " + spv.vlatex(vec_a)))

display(Markdown("---"))
display(Markdown("#### Forces on bead"))

# Force due to Earth's gravity
F_1E = m * g * (-N.y)

# Normal Force on bead by hoop 
NmH = sp.Symbol(r"N_{mH}", real=True, positive=True)  # Normal force from the hoop
F_Mh = NmH * (-Q.x).express(N)  # Force on bead

# Total force
F_total = (F_1E + F_Mh).simplify()

display(Math(r"{{}}^\mathcal {N}\vec{F}_{1E} = " + spv.vlatex(F_1E)))
display(Math(r"{{}}^\mathcal {N}\vec{F}_{mH} = " + spv.vlatex(F_Mh)))
display(Math(r"{{}}^\mathcal {N}\vec{F}_{total} = " + spv.vlatex(F_total)))


#### Position, Velocity, and Acceleration

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---

#### Forces on bead

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:

eqn1 = sp.Eq((F_total - m*vec_a).to_matrix(N).simplify(), sp.zeros(3, 1))
display(Markdown("#### Newton's Second Law:"))
display(eqn1)

#### Newton's Second Law:

⎡⎛           2    ⎞              ⎤      
⎢⎝-N_{mH} + Ω ⋅R⋅m⎠⋅sin(θ)⋅cos(ω)⎥   ⎡0⎤
⎢                                ⎥   ⎢ ⎥
⎢      N_{mH}⋅cos(θ) - g⋅m       ⎥ = ⎢0⎥
⎢                                ⎥   ⎢ ⎥
⎢⎛          2    ⎞               ⎥   ⎣0⎦
⎣⎝N_{mH} - Ω ⋅R⋅m⎠⋅sin(ω)⋅sin(θ) ⎦      

In [12]:
result_theta = sp.solve(eqn1, [NmH, theta], dict=True)
display(Markdown(rf"#### {len(result_theta)} solutions for $\theta$ and ${NmH}$:"))
display(result_theta)

display(Markdown("---"))
result_Omega = sp.solve(eqn1, [NmH, Omega], dict=True)
display(Markdown(rf"#### {len(result_Omega)} solutions for $\Omega$ and ${NmH}$:"))
display(Math(rf"\boxed{{{sp.latex(result_Omega)}}}"))

#### 3 solutions for $\theta$ and $N_{mH}$:

⎡                     ⎧         2               ⎛ g  ⎞      ⎫  ⎧         2     ↪
⎢{N_{mH}: g⋅m, θ: 0}, ⎪N_{mH}: Ω ⋅R⋅m, θ: - acos⎜────⎟ + 2⋅π⎪, ⎪N_{mH}: Ω ⋅R⋅m ↪
⎢                     ⎨                         ⎜ 2  ⎟      ⎬  ⎨               ↪
⎢                     ⎪                         ⎝Ω ⋅R⎠      ⎪  ⎪               ↪
⎣                     ⎩                                     ⎭  ⎩               ↪

↪          ⎛ g  ⎞⎫⎤
↪ , θ: acos⎜────⎟⎪⎥
↪          ⎜ 2  ⎟⎬⎥
↪          ⎝Ω ⋅R⎠⎪⎥
↪                ⎭⎦

---

#### 2 solutions for $\Omega$ and $N_{mH}$:

<IPython.core.display.Math object>